In [0]:
# Advanced PySpark Operations
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (StructType, StructField,
                                StringType, IntegerType,
                                FloatType)

In [0]:
# 1. Define schema explicitly (production practice!)
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("model", StringType(), True),
    StructField("accuracy", FloatType(), True),
    StructField("dataset_size", IntegerType(), True),
    StructField("team", StringType(), True)
])

data = [
    (1, "RF", 92.5, 10000, "TeamA"),
    (2, "XGB", 94.1, 50000, "TeamA"),
    (3, "NN", 96.3, 100000, "TeamB"),
    (4, "SVM", 88.0, 5000, "TeamB"),
    (5, "LR", 82.1, 10000, "TeamC"),
    (6, "RF_v2", 93.8, 10000, "TeamA")
]
df = spark.createDataFrame(data, schema)

In [0]:
# 2. Complex aggregations
df.groupBy("team").agg(
    F.count("*").alias("model_count"),
    F.round(F.avg("accuracy"), 2).alias("avg_accuracy"),
    F.max("accuracy").alias("best_accuracy"),
    F.sum("dataset_size").alias("total_data")
).orderBy(F.desc("avg_accuracy")).show()

+-----+-----------+------------+-------------+----------+
| team|model_count|avg_accuracy|best_accuracy|total_data|
+-----+-----------+------------+-------------+----------+
|TeamA|          3|       93.47|         94.1|     70000|
|TeamB|          2|       92.15|         96.3|    105000|
|TeamC|          1|        82.1|         82.1|     10000|
+-----+-----------+------------+-------------+----------+



In [0]:
# 3. Window functions in PySpark
window = Window.partitionBy("team").orderBy(
    F.desc("accuracy")
)
df.withColumn("rank_in_team",
    F.rank().over(window)
).withColumn("accuracy_diff_from_best",
    F.first("accuracy").over(window) - F.col("accuracy")
).show()

+---+-----+--------+------------+-----+------------+-----------------------+
| id|model|accuracy|dataset_size| team|rank_in_team|accuracy_diff_from_best|
+---+-----+--------+------------+-----+------------+-----------------------+
|  2|  XGB|    94.1|       50000|TeamA|           1|                    0.0|
|  6|RF_v2|    93.8|       10000|TeamA|           2|             0.29999542|
|  1|   RF|    92.5|       10000|TeamA|           3|              1.5999985|
|  3|   NN|    96.3|      100000|TeamB|           1|                    0.0|
|  4|  SVM|    88.0|        5000|TeamB|           2|               8.300003|
|  5|   LR|    82.1|       10000|TeamC|           1|                    0.0|
+---+-----+--------+------------+-----+------------+-----------------------+



In [0]:
# 4. UDF — User Defined Function
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

@udf(returnType=StringType())
def grade_model(accuracy):
    if accuracy >= 95: return "S-tier"
    elif accuracy >= 90: return "A-tier"
    elif accuracy >= 85: return "B-tier"
    return "C-tier"

df.withColumn("grade",
    grade_model(F.col("accuracy"))
).show()

+---+-----+--------+------------+-----+------+
| id|model|accuracy|dataset_size| team| grade|
+---+-----+--------+------------+-----+------+
|  1|   RF|    92.5|       10000|TeamA|A-tier|
|  2|  XGB|    94.1|       50000|TeamA|A-tier|
|  3|   NN|    96.3|      100000|TeamB|S-tier|
|  4|  SVM|    88.0|        5000|TeamB|B-tier|
|  5|   LR|    82.1|       10000|TeamC|C-tier|
|  6|RF_v2|    93.8|       10000|TeamA|A-tier|
+---+-----+--------+------------+-----+------+



In [0]:
# 5. Read/Write Delta Lake
df.write.format("delta").mode("overwrite").saveAsTable(
    "models_delta"
)
spark.read.table("models_delta").show()

+---+-----+--------+------------+-----+
| id|model|accuracy|dataset_size| team|
+---+-----+--------+------------+-----+
|  1|   RF|    92.5|       10000|TeamA|
|  2|  XGB|    94.1|       50000|TeamA|
|  3|   NN|    96.3|      100000|TeamB|
|  4|  SVM|    88.0|        5000|TeamB|
|  5|   LR|    82.1|       10000|TeamC|
|  6|RF_v2|    93.8|       10000|TeamA|
+---+-----+--------+------------+-----+

